In [ ]:
%%bash
set -e
CODE=$(dirname "$(find /kaggle/input -name score_noise.py | head -1)")
DATA=$(dirname "$(find /kaggle/input -name train_images.npy | head -1)")
T1=$(dirname "$(find /kaggle/input -path '*t1_noisy_ce/train_probs.npy' | head -1)")
echo "code=$CODE data=$DATA t1=$T1"; nvidia-smi -L
rm -rf /kaggle/working/cv && cp -r "$CODE" /kaggle/working/cv
ln -sfn "$DATA" /kaggle/working/data
ln -sfn "$T1" /kaggle/working/t1_noisy_ce
cd /kaggle/working/cv
python make_folds.py --data ../data --out ../folds --label_col noisy_label

In [ ]:
%%bash
# Task 3: exactly 5 runs (one per fold), 2 in parallel on the two T4s
cd /kaggle/working/cv
run() { CUDA_VISIBLE_DEVICES=$1 python train.py --data ../data --out ../outputs/t3_fold$2 --fit_idx ../folds/fit_$2.npy > ../t3_fold$2.log 2>&1; }
if [ "$(nvidia-smi -L | wc -l)" -ge 2 ]; then
  (for k in 0 2 4; do run 0 $k; done) &
  (for k in 1 3; do run 1 $k; done) &
  wait
else
  for k in 0 1 2 3 4; do run 0 $k; done
fi
for k in 0 1 2 3 4; do echo "fold $k: $(tail -n 1 ../t3_fold$k.log)"; done

In [ ]:
%%bash
set -e
cd /kaggle/working/cv
for k in 0 1 2 3 4; do test -f ../outputs/t3_fold$k/summary.json; done
python score_noise.py --data ../data --folds ../folds --fold_runs "../outputs/t3_fold{k}" --full_run ../t1_noisy_ce \
    --early_epoch 35 --out ../outputs/t3 --pred ../predictions/t3_noise_scores.csv --check
rm -f /kaggle/working/data /kaggle/working/t1_noisy_ce